In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('../data/database/DimPolitician.csv')
row = df[df.id == 30].iloc[0]

twitter_handle   = str(row["TwitterID"]).strip()              # e.g. "HolstVigilius"
instagram_handle = str(row["InstagramID"]).strip().strip("/") # e.g. "christianvigilius"
linkedin_id      = str(row["LinkedInID"]).strip().strip("/")  # e.g. "in/christian-vigilius"
linkedin_slug    = linkedin_id.split("/")[-1]                 # e.g. "christian-vigilius"

twitter_handle, instagram_handle, linkedin_id, linkedin_slug

('HolstVigilius',
 'christianvigilius',
 'in/christian-vigilius',
 'christian-vigilius')

## Extract post URLs for the 3 profiles (Twitter/X, Instagram, LinkedIn)

Instagram/X/LinkedIn block bots and require login, so we can't crawl the live
profiles. Instead we query the **Wayback Machine CDX index** for every archived
URL under each profile prefix, then keep only the ones that match a per-platform
*post* pattern.

Coverage caveats (honest):
- **Twitter/X** — tweet URLs are `.../<handle>/status/<id>`, scoped to the user, so recall is good.
- **LinkedIn** — post URLs are `linkedin.com/posts/<slug>_...`, scoped to the slug, so recall is good.
- **Instagram** — post URLs are `instagram.com/p/<code>` and are **not** scoped to a username. Wayback can only tie a post to the user when the profile-page snapshot was captured; individual `/p/` posts can't be reliably attributed by URL alone. Expect low/partial recall here.

In [9]:
import re
import time
import requests

CDX = "https://web.archive.org/cdx/search/cdx"
SESSION = requests.Session()
SESSION.headers.update(
    {"User-Agent": "OnTheRecord-research/1.0 (+contact: marcus.presutti.eu@gmail.com)"}
)


def cdx_urls(prefix, max_retries=6):
    """All distinct archived original URLs under `prefix` from the Wayback CDX index.

    A single user profile is small enough that one collapsed request works (no
    block pagination needed). A trailing `*` triggers CDX prefix matching; do
    NOT also pass matchType=prefix — the two together make CDX treat the `*`
    literally and return nothing.
    """
    params = {
        "url": f"{prefix}*",
        "output": "json",
        "fl": "original",
        "collapse": "urlkey",   # dedupe on the canonicalized URL
    }
    for attempt in range(max_retries):
        try:
            resp = SESSION.get(CDX, params=params, timeout=120)
        except (requests.ConnectionError, requests.Timeout):
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)
            continue
        if resp.status_code in (429, 503, 504):
            if attempt == max_retries - 1:
                resp.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        resp.raise_for_status()
        data = resp.json()
        return [row[0] for row in data[1:]]  # strip header row
    raise RuntimeError("unreachable")

In [10]:
def clean(url):
    """Drop Wayback prefix (if any) and trailing junk, keep the live URL."""
    url = re.sub(r"^https?://web\.archive\.org/web/\d+\w?/", "", url)
    return url.split("?")[0].rstrip("/")


# (label, [CDX prefixes to walk], compiled post-URL regex)
SOURCES = [
    ("twitter", [
        f"twitter.com/{twitter_handle}/status/",
        f"x.com/{twitter_handle}/status/",
        f"mobile.twitter.com/{twitter_handle}/status/",
    ], re.compile(rf"(?:twitter|x)\.com/{re.escape(twitter_handle)}/status/\d+", re.I)),

    # Only slug-scoped LinkedIn posts are attributable to this person.
    # linkedin.com/feed/update/ permalinks are NOT user-scoped, so we skip them.
    ("linkedin", [
        f"linkedin.com/posts/{linkedin_slug}",
    ], re.compile(rf"linkedin\.com/posts/{re.escape(linkedin_slug)}", re.I)),

    # Instagram post URLs (/p/, /reel/, /tv/) are NOT username-scoped, so only
    # posts captured under the profile prefix are attributable. Recall is
    # partial by design (see note above).
    ("instagram", [
        f"instagram.com/{instagram_handle}",
    ], re.compile(rf"instagram\.com/{re.escape(instagram_handle)}/(?:p|reel|tv)/[\w-]+", re.I)),
]

posts = {}
for label, prefixes, pat in SOURCES:
    found = set()
    for prefix in prefixes:
        for raw in cdx_urls(prefix):
            u = clean(raw)
            if pat.search(u):
                found.add(u)
        time.sleep(0.5)
    posts[label] = sorted(found)
    print(f"{label:10s} {len(posts[label])} post URLs")

twitter    485 post URLs
linkedin   0 post URLs
instagram  0 post URLs


In [11]:
out = pd.DataFrame(
    [(int(row["id"]), plat, url) for plat, urls in posts.items() for url in urls],
    columns=["politician_id", "platform", "post_url"],
)
out.to_csv(f"../data/scraped/socials/{int(row['id'])}_post_urls.csv", index=False)
print(f"wrote {len(out)} rows")
out.groupby("platform").size()

wrote 485 rows


platform
twitter    485
dtype: int64